# Simple ESG Carbon RAG (Colab-Friendly Teaching Notebook)

In this notebook, we learn how RAG works in practice with a beginner-friendly workflow.

We build a **simple RAG pipeline** to extract carbon-emissions-related information from ESG / sustainability reports for:
- Tesla
- Amazon
- Google
- Apple
- Meta

## Colab Quick Start
1. We open this notebook in Google Colab.
2. We set `Runtime -> Change runtime type -> GPU` (recommended for the local model).
3. We run the Colab setup cells first.
4. We keep `provider = "local"` unless we intentionally switch to DeepSeek/OpenAI API.

## Teaching goal
We use this flow:
1. Parse report text
2. Split into chunks
3. Store chunks in a vector database
4. Retrieve relevant chunks for a question
5. Ask an LLM to extract structured answers from retrieved evidence


## Practical Context: Green Finance / ESG Research

Analysts often need to extract:
- Scope 1 / Scope 2 / Scope 3 emissions
- Total GHG emissions
- Emissions intensity
- Carbon reduction targets
- Reporting year and boundary notes

RAG helps because ESG reports are long and noisy.
Instead of sending the whole report to an LLM, we retrieve only the most relevant passages.


## Dependencies (Colab-Friendly Setup)

For Google Colab, we run the setup cells below instead of manually installing everything.

Important:
- We usually keep Colab's preinstalled `torch` so GPU acceleration stays available.
- We install the remaining packages (including `chromadb`, `sentence-transformers`, and `transformers`) in a Colab setup cell.
- This notebook is tested locally with Python `3.12.10`, and the Colab version is designed to run in Colab's managed Python environment.


## Colab Runtime Setup (GPU-First + Packages)

We run this section first in Colab.

- We select `Runtime -> Change runtime type -> GPU` before running the notebook.
- We keep Colab's preinstalled `torch` so GPU support stays available.
- We install the remaining packages (`chromadb`, `sentence-transformers`, `transformers`, etc.).
- If Colab does not provide a GPU in the current session, the notebook prints a clear warning and we can switch runtime or use a smaller local model.


In [ ]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
USE_COLAB_GPU = True  # reminder: set GPU in Colab runtime menu
RUN_COLAB_INSTALL = True
REQUIRE_GPU_FOR_LOCAL_MODEL = True  # Colab GPU-first behavior for the local model path

print("IN_COLAB =", IN_COLAB)
print("Python version =", sys.version.split()[0])

if IN_COLAB and RUN_COLAB_INSTALL:
    packages = [
        "pypdf==6.7.3",
        "pandas==3.0.1",
        "chromadb==1.5.1",
        "sentence-transformers==5.2.3",
        "transformers==5.2.0",
        "openai==2.24.0",
        "python-dotenv==1.2.1",
        "requests==2.32.5",
        "nbformat==5.10.4",
        "nbclient==0.10.4",
        "ipykernel==7.2.0",
        "accelerate",
    ]
    print("Installing Colab packages (keeping preinstalled torch)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
    print("Colab package installation complete.")
else:
    print("Skipping Colab package installation.")

if IN_COLAB:
    print("\nGPU-first reminder:")
    print("- Select Runtime -> Change runtime type -> GPU")
    print("- Restart runtime if torch shows a CPU build")


In [ ]:
try:
    import torch
    print("torch version =", torch.__version__)
    print("CUDA available =", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU =", torch.cuda.get_device_name(0))
    else:
        print("GPU is not available in the current runtime.")
        print("If this is Colab, switch Runtime -> Change runtime type -> GPU and restart the runtime.")
except Exception as e:
    print("Torch check skipped:", e)


## Data Folder (Expected)

```text
RAG/
  simple_rag_esg_carbon_teaching.ipynb
  data/
    esg_reports/
      amazon_2024_sustainability_report.pdf
      apple_2025_environmental_progress_report.pdf
      google_2025_environmental_report.pdf
      meta_2025_sustainability_report.pdf
      tesla_2024_impact_report_extended_from_official_pdf_mirror.md   # fallback text mirror
```

Note:
- In this environment, the Tesla PDF was blocked by Tesla's CDN (Akamai 403), so we use a text mirror of the official PDF (`r.jina.ai` mirror of the Tesla PDF content).
- The parser in this notebook supports both `.pdf` and `.md/.txt`.


## Optional: Persist Files in Google Drive (Colab)

Colab local storage (`/content`) is temporary.
If we want to keep downloaded reports, vector DB files, and outputs after the session ends, we can mount Google Drive and set `PROJECT_ROOT`.


In [ ]:
USE_GOOGLE_DRIVE = False
GOOGLE_DRIVE_PROJECT_DIR = "/content/drive/MyDrive/ESG_RAG_colab"
import sys

if "google.colab" in sys.modules and USE_GOOGLE_DRIVE:
    from google.colab import drive
    import os
    drive.mount("/content/drive")
    os.makedirs(GOOGLE_DRIVE_PROJECT_DIR, exist_ok=True)
    os.environ["PROJECT_ROOT"] = GOOGLE_DRIVE_PROJECT_DIR
    print("PROJECT_ROOT set to:", os.environ["PROJECT_ROOT"])
else:
    print("Using current working directory. PROJECT_ROOT is not overridden.")


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys
import re
import json
import time
from datetime import datetime
from typing import Any, Dict, List

import pandas as pd
import requests
from pypdf import PdfReader
from dotenv import load_dotenv

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

load_dotenv()

IN_COLAB = "google.colab" in sys.modules
if "REQUIRE_GPU_FOR_LOCAL_MODEL" not in os.environ:
    os.environ["REQUIRE_GPU_FOR_LOCAL_MODEL"] = "1" if IN_COLAB else "0"
PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT", ".")).resolve()
DATA_DIR = PROJECT_ROOT / "data" / "esg_reports"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
VECTOR_DB_DIR = PROJECT_ROOT / "vector_db" / "chroma_esg"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
VECTOR_DB_DIR.mkdir(exist_ok=True, parents=True)

print("IN_COLAB:", IN_COLAB)
print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR.resolve())
print("Output dir:", OUTPUT_DIR.resolve())
print("Vector DB dir:", VECTOR_DB_DIR.resolve())


## Optional Download Helper (for reproducibility)

We use these URLs in this run.
In Colab, we keep the remote download option and default to downloading missing files automatically.
We can still set `DOWNLOAD_MISSING = False` if the files are already available.

Important:
- Tesla direct PDF may return `403 Forbidden` from some networks.
- This helper therefore includes a **Tesla text mirror fallback** (`r.jina.ai`) for classroom use.


In [ ]:
DOWNLOAD_MISSING = bool(globals().get("IN_COLAB", False))
# Colab default: download missing reports automatically.
# Local default: keep False unless we want to download.
# We can override manually, for example:
# DOWNLOAD_MISSING = False

SOURCE_URLS = {
    "amazon_2024_sustainability_report.pdf": "https://sustainability.aboutamazon.com/2024-amazon-sustainability-report.pdf",
    "apple_2025_environmental_progress_report.pdf": "https://www.apple.com/environment/pdf/Apple_Environmental_Progress_Report_2025.pdf",
    "google_2025_environmental_report.pdf": "https://www.gstatic.com/gumdrop/sustainability/google-2025-environmental-report.pdf",
    "meta_2025_sustainability_report.pdf": "https://sustainability.atmeta.com/wp-content/uploads/2025/08/Meta_2025-Sustainability-Report_.pdf",
    "tesla_2024_impact_report_extended_from_official_pdf_mirror.md": "https://r.jina.ai/http://www.tesla.com/ns_videos/2024-extended-version-tesla-impact-report.pdf",
}

if DOWNLOAD_MISSING:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    headers = {"User-Agent": "Mozilla/5.0"}
    for name, url in SOURCE_URLS.items():
        path = DATA_DIR / name
        if path.exists() and path.stat().st_size > 100_000:
            print("SKIP", name)
            continue
        print("Downloading", name)
        r = requests.get(url, headers=headers, timeout=300)
        r.raise_for_status()
        if name.endswith(".pdf"):
            path.write_bytes(r.content)
        else:
            path.write_text(r.text, encoding="utf-8")
        print("Saved", name, path.stat().st_size, "bytes")
else:
    print("DOWNLOAD_MISSING = False (using local files if present)")

with open(OUTPUT_DIR / "download_sources.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "generated_at": datetime.utcnow().isoformat() + "Z",
            "source_urls": SOURCE_URLS,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )
print("Saved source URL record to", OUTPUT_DIR / "download_sources.json")


## Step 1. Load Documents (PDF + Text Mirror Support)

We keep parsing lightweight:
- `pypdf` for normal PDFs
- direct text loading for `.md/.txt`

For the Tesla mirror file, we remove the `r.jina.ai` header and keep the report content section.


In [ ]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    parts = []
    for page in reader.pages:
        parts.append(page.extract_text() or "")
    return "\n".join(parts)


def extract_text_from_text_file(path: Path) -> str:
    text = path.read_text(encoding="utf-8", errors="ignore")
    # r.jina.ai PDF mirror format includes a header and then "Markdown Content:"
    marker = "Markdown Content:"
    if marker in text:
        text = text.split(marker, 1)[1].strip()
    return text


def infer_company_label(file_name: str) -> str:
    lower = file_name.lower()
    if lower.startswith("tesla_"):
        return "Tesla"
    if lower.startswith("amazon_"):
        return "Amazon"
    if lower.startswith("google_"):
        return "Google"
    if lower.startswith("apple_"):
        return "Apple"
    if lower.startswith("meta_"):
        return "Meta"
    return Path(file_name).stem


def load_documents(data_dir: Path) -> List[Dict[str, Any]]:
    docs: List[Dict[str, Any]] = []
    for path in sorted(data_dir.iterdir()):
        if path.suffix.lower() not in {".pdf", ".md", ".txt"}:
            continue
        try:
            if path.suffix.lower() == ".pdf":
                text = extract_text_from_pdf(path)
                source_type = "pdf"
            else:
                text = extract_text_from_text_file(path)
                source_type = "text_mirror"
        except Exception as e:
            print(f"SKIP (parse error): {path.name} -> {e}")
            continue

        docs.append(
            {
                "doc_id": path.stem,
                "company": infer_company_label(path.name),
                "file_name": path.name,
                "source_type": source_type,
                "text": text,
                "char_count": len(text),
            }
        )
    return docs


documents = load_documents(DATA_DIR) if DATA_DIR.exists() else []
print(f"Loaded {len(documents)} document(s).")
for d in documents:
    print(f"- {d['company']}: {d['file_name']} ({d['source_type']}) | {d['char_count']:,} chars")


## Step 2. Chunk the Documents

We split report text into chunks before storing them in the vector database.

This is still a simple, teaching-friendly chunker:
- whitespace cleanup
- paragraph-aware chunking
- small overlap


In [ ]:
def normalize_text(text: str) -> str:
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def chunk_text(text: str, max_chars: int = 900, overlap_chars: int = 120) -> List[str]:
    text = normalize_text(text)
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks: List[str] = []
    current = ""

    for para in paragraphs:
        candidate = f"{current}\n\n{para}".strip() if current else para
        if len(candidate) <= max_chars:
            current = candidate
            continue

        if current:
            chunks.append(current)
            current = ""

        if len(para) <= max_chars:
            current = para
        else:
            start = 0
            while start < len(para):
                end = start + max_chars
                piece = para[start:end]
                chunks.append(piece)
                if end >= len(para):
                    break
                start = max(0, end - overlap_chars)

    if current:
        chunks.append(current)

    # lightweight overlap between adjacent chunks
    if overlap_chars > 0 and len(chunks) > 1:
        out = [chunks[0]]
        for i in range(1, len(chunks)):
            prefix = chunks[i - 1][-overlap_chars:]
            out.append((prefix + "\n" + chunks[i]).strip())
        chunks = out

    return chunks


def build_chunk_records(docs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    for doc in docs:
        chunks = chunk_text(doc["text"])
        for i, chunk in enumerate(chunks):
            records.append(
                {
                    "id": f"{doc['doc_id']}::chunk::{i}",
                    "doc_id": doc["doc_id"],
                    "company": doc["company"],
                    "file_name": doc["file_name"],
                    "source_type": doc["source_type"],
                    "chunk_id": i,
                    "text": chunk,
                }
            )
    return records


chunk_records = build_chunk_records(documents)
print("Total chunks:", len(chunk_records))
if chunk_records:
    print("Example chunk:", chunk_records[0]["id"])
    print(chunk_records[0]["text"][:600])


## Step 3. Build a Vector Database (ChromaDB)

Now we store chunks in a vector database using:
- `ChromaDB` (local persistent vector DB)
- `all-MiniLM-L6-v2` embeddings (small and practical for classroom use)


In [ ]:
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "esg_carbon_chunks_course_v1"
REBUILD_VECTOR_DB = True  # set False if you want to reuse an existing collection

embedding_fn = SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL_NAME)
chroma_client = chromadb.PersistentClient(path=str(VECTOR_DB_DIR))

if REBUILD_VECTOR_DB:
    try:
        chroma_client.delete_collection(COLLECTION_NAME)
        print("Deleted existing collection:", COLLECTION_NAME)
    except Exception:
        pass

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)


def index_chunks_to_chroma(collection, chunk_records: List[Dict[str, Any]], batch_size: int = 64) -> None:
    if not chunk_records:
        print("No chunks to index.")
        return

    for start in range(0, len(chunk_records), batch_size):
        batch = chunk_records[start : start + batch_size]
        collection.add(
            ids=[r["id"] for r in batch],
            documents=[r["text"] for r in batch],
            metadatas=[
                {
                    "doc_id": r["doc_id"],
                    "company": r["company"],
                    "file_name": r["file_name"],
                    "source_type": r["source_type"],
                    "chunk_id": int(r["chunk_id"]),
                }
                for r in batch
            ],
        )
    print("Indexed", len(chunk_records), "chunks into ChromaDB.")


index_chunks_to_chroma(collection, chunk_records)
print("Collection count:", collection.count())


## Step 4. Retrieval from the Vector DB

This function retrieves the most relevant chunks for a question.
We can also filter to a single company/report using metadata (`doc_id`).


In [ ]:
def retrieve_chunks(query: str, top_k: int = 4, filter_doc_id: str | None = None) -> List[Dict[str, Any]]:
    kwargs: Dict[str, Any] = {
        "query_texts": [query],
        "n_results": top_k,
        "include": ["documents", "metadatas", "distances"],
    }
    if filter_doc_id is not None:
        kwargs["where"] = {"doc_id": filter_doc_id}

    res = collection.query(**kwargs)

    docs = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    distances = res.get("distances", [[]])[0]

    out = []
    for doc_text, meta, distance in zip(docs, metas, distances):
        item = dict(meta)
        item["text"] = doc_text
        item["distance"] = float(distance)
        # cosine distance -> rough similarity (teaching convenience)
        item["similarity"] = max(0.0, 1.0 - float(distance))
        out.append(item)
    return out


example_query = "Scope 1 Scope 2 Scope 3 emissions total GHG emissions emissions intensity carbon targets reporting year"
if documents:
    example_doc_id = documents[0]["doc_id"]
    hits = retrieve_chunks(example_query, top_k=3, filter_doc_id=example_doc_id)
    print("Retrieved chunks for:", example_doc_id)
    for i, h in enumerate(hits, 1):
        print(f"\n[{i}] {h['file_name']} | chunk={h['chunk_id']} | similarity={h['similarity']:.4f}")
        print(h["text"][:500])
else:
    print("No documents found.")


## Step 5. LLM Backends (Local Distilled Model / DeepSeek / ChatGPT)

We provide a **single extraction pipeline** and make the LLM backend swappable.

### Backend A: Local distilled model (default in this notebook)
- Suggested model: `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`
- Runs locally (Colab GPU recommended; CPU fallback is much slower)

### Backend B: DeepSeek API
- Set `DEEPSEEK_API_KEY`

### Backend C: ChatGPT / OpenAI API
- Set `OPENAI_API_KEY`

We keep retrieval fixed and swap only the generation backend.


In [ ]:
from openai import OpenAI

_LOCAL_LLM_CACHE: Dict[str, Any] = {}


def _local_runtime_config() -> Dict[str, Any]:
    import torch

    if torch.cuda.is_available():
        if hasattr(torch.cuda, "is_bf16_supported") and torch.cuda.is_bf16_supported():
            dtype = torch.bfloat16
            dtype_name = "bfloat16"
        else:
            dtype = torch.float16
            dtype_name = "float16"
        return {"device_type": "cuda", "dtype": dtype, "dtype_name": dtype_name}
    return {"device_type": "cpu", "dtype": torch.float32, "dtype_name": "float32"}


def get_local_llm(model_id: str | None = None):
    """
    Load a local instruction model via transformers.
    This version is Colab-friendly and uses GPU automatically when available.
    """
    model_id = model_id or os.getenv("LOCAL_MODEL_ID", "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
    if model_id in _LOCAL_LLM_CACHE:
        return _LOCAL_LLM_CACHE[model_id]

    print(f"Loading local model: {model_id}")
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    runtime_cfg = _local_runtime_config()
    in_colab = "google.colab" in sys.modules
    require_gpu = os.getenv("REQUIRE_GPU_FOR_LOCAL_MODEL", "0") == "1"

    print("Local model device =", runtime_cfg["device_type"], "| dtype =", runtime_cfg["dtype_name"])
    if runtime_cfg["device_type"] == "cuda":
        print("GPU =", torch.cuda.get_device_name(0))
        # Small speedups on common Colab GPUs
        try:
            torch.backends.cuda.matmul.allow_tf32 = True
        except Exception:
            pass
        try:
            torch.set_float32_matmul_precision("high")
        except Exception:
            pass
    elif in_colab and require_gpu:
        raise RuntimeError(
            "Colab GPU-first mode is enabled, but CUDA is not available. "
            "Switch Colab runtime to GPU and restart the runtime, or set "
            "REQUIRE_GPU_FOR_LOCAL_MODEL=0 to allow CPU fallback."
        )

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs: Dict[str, Any] = {"torch_dtype": runtime_cfg["dtype"]}
    if runtime_cfg["device_type"] == "cuda":
        model_kwargs["device_map"] = "auto"
        model_kwargs["low_cpu_mem_usage"] = True

    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    if runtime_cfg["device_type"] == "cpu":
        model = model.to("cpu")

    model.eval()
    _LOCAL_LLM_CACHE[model_id] = (tokenizer, model, runtime_cfg)
    return tokenizer, model, runtime_cfg


def call_local_llm(
    system_prompt: str,
    user_prompt: str,
    model_id: str | None = None,
    max_new_tokens: int | None = None,
) -> str:
    tokenizer, model, runtime_cfg = get_local_llm(model_id=model_id)
    import torch

    max_new_tokens = max_new_tokens or int(os.getenv("LOCAL_MAX_NEW_TOKENS", "500"))

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    if hasattr(tokenizer, "apply_chat_template"):
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        prompt_text = f"System:\n{system_prompt}\n\nUser:\n{user_prompt}\n\nAssistant:\n"

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=3072,
    )

    if runtime_cfg["device_type"] == "cuda":
        model_device = next(model.parameters()).device
        inputs = {k: v.to(model_device) for k, v in inputs.items()}

    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=pad_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_tokens = outputs[0][inputs["input_ids"].shape[-1] :]
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()


def get_api_client(provider: str):
    provider = provider.lower().strip()
    if provider == "openai":
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("Missing OPENAI_API_KEY")
        return OpenAI(api_key=api_key), os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    if provider == "deepseek":
        api_key = os.getenv("DEEPSEEK_API_KEY")
        if not api_key:
            raise ValueError("Missing DEEPSEEK_API_KEY")
        base_url = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com")
        return OpenAI(api_key=api_key, base_url=base_url), os.getenv("DEEPSEEK_MODEL", "deepseek-chat")
    raise ValueError("provider must be 'openai' or 'deepseek'")


def call_api_llm(provider: str, system_prompt: str, user_prompt: str, model: str | None = None) -> str:
    client, default_model = get_api_client(provider)
    model = model or default_model
    resp = client.chat.completions.create(
        model=model,
        temperature=0.0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return resp.choices[0].message.content or ""


def call_llm(provider: str, system_prompt: str, user_prompt: str, model: str | None = None) -> str:
    provider = provider.lower().strip()
    if provider == "local":
        return call_local_llm(system_prompt, user_prompt, model_id=model)
    if provider in {"openai", "deepseek"}:
        return call_api_llm(provider, system_prompt, user_prompt, model=model)
    raise ValueError("provider must be one of: local, openai, deepseek")


## Step 6. Build the Extraction Prompt

We ask the model to return JSON only.
This makes it easier to convert results into a table for analysis.


In [ ]:
SYSTEM_PROMPT = """
You are an ESG research assistant.
Extract carbon-emissions-related information from report excerpts.
Use only the provided excerpts.
Return JSON only.
If a field is missing, use null.
Do not invent numbers.
""".strip()

EXTRACTION_SCHEMA = {
    "company": "string or null",
    "report_year": "string or null",
    "scope_1_emissions": "string or null",
    "scope_2_emissions": "string or null",
    "scope_3_emissions": "string or null",
    "total_ghg_emissions": "string or null",
    "emissions_intensity": "string or null",
    "target_summary": "string or null",
    "evidence_quotes": ["short quotes/snippets from retrieved excerpts"],
    "confidence": "low | medium | high",
    "notes": "ambiguity / unit / boundary notes",
}


def build_extraction_prompt(company_name: str, query: str, chunks: List[Dict[str, Any]]) -> str:
    blocks = []
    for i, c in enumerate(chunks, 1):
        blocks.append(
            f"[Chunk {i}] file={c['file_name']} chunk_id={c['chunk_id']} similarity={c['similarity']:.4f}\n{c['text']}"
        )

    return f"""
Task: Extract carbon emissions information for {company_name}.

Question:
{query}

Output JSON schema (use these exact keys):
{json.dumps(EXTRACTION_SCHEMA, indent=2)}

Retrieved report excerpts:
{chr(10).join([''] + [b + chr(10) for b in blocks])}

Rules:
1. Use only retrieved excerpts.
2. Preserve units as written.
3. If multiple years appear, choose the most likely reporting year and explain ambiguity in notes.
4. Return JSON only (no markdown).
""".strip()


In [ ]:
def extract_first_json_object(text: str) -> str:
    text = text.strip()
    # Fast path
    if text.startswith("{") and text.endswith("}"):
        return text

    # Find first balanced JSON object
    start = text.find("{")
    if start == -1:
        raise ValueError("No JSON object start found")

    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start : i + 1]
    raise ValueError("No balanced JSON object found")


def parse_model_json(raw_text: str) -> Dict[str, Any]:
    try:
        return json.loads(raw_text)
    except Exception:
        candidate = extract_first_json_object(raw_text)
        return json.loads(candidate)


## Step 7. Single-Company RAG Extraction Function

This function combines:
- vector retrieval (ChromaDB)
- prompt building
- LLM call (local or API)
- JSON parsing


In [ ]:
RAG_QUERY = (
    "Find carbon emissions information including Scope 1, Scope 2, Scope 3, total GHG emissions, "
    "emissions intensity, reporting year, and emissions reduction targets."
)


def rag_extract_for_doc(
    doc: Dict[str, Any],
    provider: str = "local",
    model: str | None = None,
    top_k: int = 4,
) -> Dict[str, Any]:
    hits = retrieve_chunks(RAG_QUERY, top_k=top_k, filter_doc_id=doc["doc_id"])
    if not hits:
        return {"company": doc["company"], "error": f"No hits for {doc['doc_id']}"}

    user_prompt = build_extraction_prompt(doc["company"], RAG_QUERY, hits)
    t0 = time.time()
    raw = call_llm(provider=provider, system_prompt=SYSTEM_PROMPT, user_prompt=user_prompt, model=model)
    elapsed = time.time() - t0

    try:
        data = parse_model_json(raw)
    except Exception as e:
        data = {
            "company": doc["company"],
            "parse_error": str(e),
            "raw_model_output": raw[:5000],
        }

    data["_rag_meta"] = {
        "provider": provider,
        "model": model or ("LOCAL_MODEL_ID env / default" if provider == "local" else "provider default"),
        "doc_id": doc["doc_id"],
        "file_name": doc["file_name"],
        "source_type": doc["source_type"],
        "top_k": top_k,
        "latency_sec": round(elapsed, 2),
        "retrieved_chunks": [
            {
                "chunk_id": h["chunk_id"],
                "similarity": h["similarity"],
                "file_name": h["file_name"],
            }
            for h in hits
        ],
    }
    return data


## Step 8. Configure the Backend for This Run

We start with a **local distilled model** by default.

If the machine is slow or RAM is limited, we can set:
- `LOCAL_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"` (smaller fallback, not distilled)

API alternatives (optional; these do not run unless we change `provider`):
- `provider = "deepseek"`
- `provider = "openai"`

Colab GPU recommendation:
- We keep `provider = "local"` and run on GPU.
- If Colab does not provide a GPU, we can switch runtime or temporarily use the smaller local model.


In [ ]:
# Choose backend: "local", "deepseek", or "openai"
# We keep "local" by default so API backends remain optional and do not run.
provider = "local"

# Colab GPU-first behavior for the local model:
# - In Colab, the notebook expects GPU by default (REQUIRE_GPU_FOR_LOCAL_MODEL=1)
# - To allow CPU fallback manually (much slower), set:
#     import os; os.environ["REQUIRE_GPU_FOR_LOCAL_MODEL"] = "0"

# Local default: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
# Smaller fallback (faster / easier on free Colab GPU, but weaker quality):
# model = "Qwen/Qwen2.5-0.5B-Instruct"
model = None

top_k = 4

print("provider =", provider)
print("model =", model)
print("REQUIRE_GPU_FOR_LOCAL_MODEL =", os.getenv("REQUIRE_GPU_FOR_LOCAL_MODEL", "(unset)"))
print("documents =", [d["company"] for d in documents])


In [ ]:
try:
    import torch
    print("torch version =", torch.__version__)
    print("CUDA available =", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU =", torch.cuda.get_device_name(0))
    else:
        print("GPU is not available in the current runtime.")
        print("If this is Colab, switch Runtime -> Change runtime type -> GPU and restart the runtime.")
except Exception as e:
    print("Torch check skipped:", e)


## Step 9. Test One Company First (Recommended)

Run one company first to confirm the pipeline works before batch mode.


In [ ]:
RUN_ONE_COMPANY = True

single_result = None
if RUN_ONE_COMPANY and documents:
    # Start with Tesla if available (using mirrored text file in this environment)
    preferred_order = ["Tesla", "Amazon", "Google", "Apple", "Meta"]
    selected = None
    for name in preferred_order:
        selected = next((d for d in documents if d["company"] == name), None)
        if selected:
            break

    print("Running one-company run for:", selected["company"], "|", selected["file_name"])
    single_result = rag_extract_for_doc(selected, provider=provider, model=model, top_k=top_k)
    print(json.dumps(single_result, indent=2, ensure_ascii=False)[:4000])
else:
    print("Skipping one-company run.")


## Step 10. Batch Run on Tesla / Amazon / Google / Apple / Meta

We run the full pipeline across the five-company set and save outputs.


In [ ]:
TARGET_COMPANIES = ["Tesla", "Amazon", "Google", "Apple", "Meta"]
RUN_BATCH = True


def batch_extract(
    docs: List[Dict[str, Any]],
    provider: str = "local",
    model: str | None = None,
    top_k: int = 4,
    target_companies: List[str] | None = None,
) -> List[Dict[str, Any]]:
    selected_docs = docs
    if target_companies:
        target_set = set(target_companies)
        selected_docs = [d for d in docs if d["company"] in target_set]

    results = []
    for doc in selected_docs:
        print(f"\n=== Processing {doc['company']} | {doc['file_name']} ===")
        try:
            result = rag_extract_for_doc(doc, provider=provider, model=model, top_k=top_k)
        except Exception as e:
            result = {
                "company": doc["company"],
                "error": str(e),
                "_rag_meta": {
                    "doc_id": doc["doc_id"],
                    "file_name": doc["file_name"],
                    "source_type": doc["source_type"],
                },
            }
        results.append(result)
    return results


all_results: List[Dict[str, Any]] = []
if RUN_BATCH:
    all_results = batch_extract(
        documents,
        provider=provider,
        model=model,
        top_k=top_k,
        target_companies=TARGET_COMPANIES,
    )
    print("\nBatch complete:", len(all_results), "result(s)")
else:
    print("RUN_BATCH = False")


## Step 11. Convert Results to a Table

This makes the extraction output easier to inspect and compare across firms.


In [ ]:
summary_df = pd.DataFrame(
    [
        {
            "company": r.get("company"),
            "report_year": r.get("report_year"),
            "scope_1_emissions": r.get("scope_1_emissions"),
            "scope_2_emissions": r.get("scope_2_emissions"),
            "scope_3_emissions": r.get("scope_3_emissions"),
            "total_ghg_emissions": r.get("total_ghg_emissions"),
            "emissions_intensity": r.get("emissions_intensity"),
            "target_summary": r.get("target_summary"),
            "confidence": r.get("confidence"),
            "notes": r.get("notes"),
            "error": r.get("error"),
            "parse_error": r.get("parse_error"),
            "latency_sec": (r.get("_rag_meta") or {}).get("latency_sec"),
            "source_type": (r.get("_rag_meta") or {}).get("source_type"),
            "file_name": (r.get("_rag_meta") or {}).get("file_name"),
        }
        for r in all_results
    ]
)

summary_df


## Step 12. Save Outputs and Run Record

We save:
- JSON results
- CSV summary table
- a run-record JSON (backend, model, timestamp, files used)


In [ ]:
run_timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

results_json_path = OUTPUT_DIR / "esg_carbon_rag_results.json"
summary_csv_path = OUTPUT_DIR / "esg_carbon_rag_results.csv"
run_record_path = OUTPUT_DIR / f"notebook_run_record_{run_timestamp}.json"

with open(results_json_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

summary_df.to_csv(summary_csv_path, index=False)

run_record = {
    "run_timestamp_utc": run_timestamp,
    "provider": provider,
    "model": model,
    "top_k": top_k,
    "target_companies": TARGET_COMPANIES,
    "doc_count_loaded": len(documents),
    "doc_files": [d["file_name"] for d in documents],
    "vector_db_path": str(VECTOR_DB_DIR),
    "vector_collection": COLLECTION_NAME,
    "embedding_model": EMBED_MODEL_NAME,
    "results_json_path": str(results_json_path),
    "summary_csv_path": str(summary_csv_path),
}

with open(run_record_path, "w", encoding="utf-8") as f:
    json.dump(run_record, f, indent=2, ensure_ascii=False)

print("Saved:", results_json_path)
print("Saved:", summary_csv_path)
print("Saved:", run_record_path)
